In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Ensure timestamp numeric and create timezone-aware datetime columns (UTC -> Asia/Shanghai)
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
        df['dt'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['dt'] = pd.NaT

# Helper to compute summaries for metric/trace/log files
def compute_component_name_agg(df, comp_col, name_col):
    # ensure value is numeric
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    grouped = df.groupby([comp_col, name_col])['value']
    agg = grouped.agg(
        count='count',
        min='min',
        p50=lambda x: x.quantile(0.5),
        mean='mean',
        p95=lambda x: x.quantile(0.95),
        max='max'
    ).reset_index()
    agg = agg.sort_values('count', ascending=False).reset_index(drop=True)
    return agg

# --- metric file summaries ---
metric_cmdb_count = metric_df['cmdb_id'].nunique()
metric_cmdb_samples = list(pd.Series(metric_df['cmdb_id'].dropna().unique())[:20])
metric_kpi_count = metric_df['kpi_name'].nunique() if 'kpi_name' in metric_df.columns else 0
metric_kpi_samples = list(pd.Series(metric_df['kpi_name'].dropna().unique())[:20]) if 'kpi_name' in metric_df.columns else []

metric_agg_full = compute_component_name_agg(metric_df, 'cmdb_id', 'kpi_name')
metric_agg_top50 = metric_agg_full.head(50)

metric_time_range = pd.DataFrame({
    'file': ['metric.csv'],
    'earliest': [metric_df['dt'].min()],
    'latest': [metric_df['dt'].max()]
})

# Compact summaries as small DataFrames
metric_cmdb_summary = pd.DataFrame({
    'unique_cmdb_count': [metric_cmdb_count],
    'sample_cmdb_ids': [metric_cmdb_samples]
})
metric_kpi_summary = pd.DataFrame({
    'unique_kpi_count': [metric_kpi_count],
    'sample_kpi_names': [metric_kpi_samples]
})

# --- trace file summaries ---
trace_cmdb_count = trace_df['cmdb_id'].nunique()
trace_cmdb_samples = list(pd.Series(trace_df['cmdb_id'].dropna().unique())[:20])
trace_kpi_count = trace_df['trace_name'].nunique() if 'trace_name' in trace_df.columns else 0
trace_kpi_samples = list(pd.Series(trace_df['trace_name'].dropna().unique())[:20]) if 'trace_name' in trace_df.columns else []

# rename trace column to generic name for aggregation function
trace_df = trace_df.rename(columns={'trace_name': 'trace_name'})  # keep name; compute function will use provided name_col
trace_agg_full = compute_component_name_agg(trace_df.rename(columns={'trace_name': 'kpi_name'}), 'cmdb_id', 'kpi_name')
trace_agg_top50 = trace_agg_full.head(50)

trace_time_range = pd.DataFrame({
    'file': ['trace.csv'],
    'earliest': [trace_df['dt'].min()],
    'latest': [trace_df['dt'].max()]
})

trace_cmdb_summary = pd.DataFrame({
    'unique_cmdb_count': [trace_cmdb_count],
    'sample_cmdb_ids': [trace_cmdb_samples]
})
trace_kpi_summary = pd.DataFrame({
    'unique_trace_count': [trace_kpi_count],
    'sample_trace_names': [trace_kpi_samples]
})

# --- log file summaries ---
log_cmdb_count = log_df['cmdb_id'].nunique()
log_cmdb_samples = list(pd.Series(log_df['cmdb_id'].dropna().unique())[:20])
log_kpi_count = log_df['log_name'].nunique() if 'log_name' in log_df.columns else 0
log_kpi_samples = list(pd.Series(log_df['log_name'].dropna().unique())[:20]) if 'log_name' in log_df.columns else []

log_df = log_df.rename(columns={'log_name': 'kpi_name'})
log_agg_full = compute_component_name_agg(log_df, 'cmdb_id', 'kpi_name')
log_agg_top50 = log_agg_full.head(50)

log_time_range = pd.DataFrame({
    'file': ['log.csv'],
    'earliest': [log_df['dt'].min()],
    'latest': [log_df['dt'].max()]
})

log_cmdb_summary = pd.DataFrame({
    'unique_cmdb_count': [log_cmdb_count],
    'sample_cmdb_ids': [log_cmdb_samples]
})
log_kpi_summary = pd.DataFrame({
    'unique_log_count': [log_kpi_count],
    'sample_log_names': [log_kpi_samples]
})

# --- error_logs summaries ---
error_total_count = len(error_df)
error_counts_top20 = error_df.groupby('cmdb_id').size().sort_values(ascending=False).head(20).reset_index(name='count')

top3_cmdb = list(error_counts_top20['cmdb_id'].head(3))
# up to 5 sample messages per top-3 cmdb_id
def sample_messages_for_cmdb(cid):
    msgs = error_df.loc[error_df['cmdb_id'] == cid, 'message'].dropna().astype(str).unique()[:5]
    return list(msgs)

error_samples_top3 = pd.DataFrame({
    'cmdb_id': top3_cmdb,
    'sample_messages': [sample_messages_for_cmdb(cid) for cid in top3_cmdb]
})

error_time_range = pd.DataFrame({
    'file': ['error_logs.csv'],
    'earliest': [error_df['dt'].min()],
    'latest': [error_df['dt'].max()]
})

# Final compact outputs (limited to top results and small summaries)
metric_cmdb_summary, metric_kpi_summary, metric_agg_top50, metric_time_range, \
trace_cmdb_summary, trace_kpi_summary, trace_agg_top50, trace_time_range, \
log_cmdb_summary, log_kpi_summary, log_agg_top50, log_time_range, \
error_total_count, error_counts_top20, error_samples_top3, error_time_range

```
Out[1]:
```
```python
# Summarize previously computed aggregates into a concise plain-English summary string.
# Reuse existing kernel variables: metric_cmdb_summary, metric_kpi_summary, metric_agg_top50, metric_time_range,
# trace_cmdb_summary, trace_kpi_summary, trace_agg_top50, trace_time_range,
# log_cmdb_summary, log_kpi_summary, log_agg_top50, log_time_range,
# error_total_count, error_counts_top20, error_samples_top3, error_time_range

def first_n_list_from_df(df_col, n=5):
    vals = list(df_col) if hasattr(df_col, '__iter__') else []
    return vals[:n]

# Metric summaries
m_cmdb_count = int(metric_cmdb_summary.loc[0, 'unique_cmdb_count'])
m_cmdb_samples = metric_cmdb_summary.loc[0, 'sample_cmdb_ids']
m_kpi_count = int(metric_kpi_summary.loc[0, 'unique_kpi_count'])
m_kpi_samples = metric_kpi_summary.loc[0, 'sample_kpi_names']
m_time_earliest = str(metric_time_range.loc[0, 'earliest'])
m_time_latest = str(metric_time_range.loc[0, 'latest'])
m_top3 = metric_agg_top50.head(3)

# Trace summaries
t_cmdb_count = int(trace_cmdb_summary.loc[0, 'unique_cmdb_count'])
t_cmdb_samples = trace_cmdb_summary.loc[0, 'sample_cmdb_ids']
t_kpi_count = int(trace_kpi_summary.loc[0, 'unique_trace_count'])
t_kpi_samples = trace_kpi_summary.loc[0, 'sample_trace_names']
t_time_earliest = str(trace_time_range.loc[0, 'earliest'])
t_time_latest = str(trace_time_range.loc[0, 'latest'])
t_top3 = trace_agg_top50.head(3)

# Log summaries
l_cmdb_count = int(log_cmdb_summary.loc[0, 'unique_cmdb_count'])
l_cmdb_samples = log_cmdb_summary.loc[0, 'sample_cmdb_ids']
l_kpi_count = int(log_kpi_summary.loc[0, 'unique_log_count'])
l_kpi_samples = log_kpi_summary.loc[0, 'sample_log_names']
l_time_earliest = str(log_time_range.loc[0, 'earliest'])
l_time_latest = str(log_time_range.loc[0, 'latest'])
l_top3 = log_agg_top50.head(3)

# Error logs summaries
e_total = int(error_total_count)
e_top_counts = [(row['cmdb_id'], int(row['count'])) for _, row in error_counts_top20.head(3).iterrows()]
e_samples = {row['cmdb_id']: row['sample_messages'] for _, row in error_samples_top3.iterrows()}
e_time_earliest = str(error_time_range.loc[0, 'earliest'])
e_time_latest = str(error_time_range.loc[0, 'latest'])

# Helper to format top rows compactly
def format_top_rows(df):
    lines = []
    for _, r in df.iterrows():
        lines.append(f"{r['cmdb_id']} | {r['kpi_name']} | count={int(r['count'])} | min={r['min']:.6g} p50={r['p50']:.6g} mean={r['mean']:.6g} p95={r['p95']:.6g} max={r['max']:.6g}")
    return lines

m_top_lines = format_top_rows(m_top3)
t_top_lines = format_top_rows(t_top3)
l_top_lines = format_top_rows(l_top3)

summary = (
    "Summary of telemetry files (compact):\n\n"
    "1) metric.csv\n"
    f"- Unique components (cmdb_id): {m_cmdb_count}. Sample up to 20: {m_cmdb_samples[:20]}\n"
    f"- Unique KPI names: {m_kpi_count}. Sample up to 20: {m_kpi_samples[:20]}\n"
    f"- Time range (UTC+8): earliest={m_time_earliest}, latest={m_time_latest}\n"
    "- Top 3 component-KPI rows by point count (component | kpi | count | min p50 mean p95 max):\n"
    + "\n".join("  - " + s for s in m_top_lines) + "\n\n"
    "2) trace.csv\n"
    f"- Unique components (cmdb_id): {t_cmdb_count}. Sample up to 20: {t_cmdb_samples[:20]}\n"
    f"- Unique trace names: {t_kpi_count}. Sample up to 20: {t_kpi_samples[:20]}\n"
    f"- Time range (UTC+8): earliest={t_time_earliest}, latest={t_time_latest}\n"
    "- Top 3 component-trace rows by point count (component | trace | count | min p50 mean p95 max):\n"
    + "\n".join("  - " + s for s in t_top_lines) + "\n\n"
    "3) log.csv\n"
    f"- Unique components (cmdb_id): {l_cmdb_count}. Sample up to 20: {l_cmdb_samples[:20]}\n"
    f"- Unique log names: {l_kpi_count}. Sample up to 20: {l_kpi_samples[:20]}\n"
    f"- Time range (UTC+8): earliest={l_time_earliest}, latest={l_time_latest}\n"
    "- Top 3 component-log rows by point count (component | log | count | min p50 mean p95 max):\n"
    + "\n".join("  - " + s for s in l_top_lines) + "\n\n"
    "4) error_logs.csv\n"
    f"- Total rows: {e_total}\n"
    f"- Top error counts by component (top 3): {e_top_counts}\n"
    f"- Sample messages (up to 5) for those top-3 components:\n"
    + "\n".join(f"  - {cid}: {e_samples.get(cid, [])}" for cid, _ in e_top_counts) + "\n"
    f"- Time range (UTC+8): earliest={e_time_earliest}, latest={e_time_latest}\n\n"
    "Notes:\n"
    "- Component-KPI aggregates were computed on the full series (count, min, median (p50), mean, p95, max).\n"
    "- Only compact top rows and samples are shown here; full aggregate tables are stored in kernel variables "
    "(metric_agg_full, trace_agg_full, log_agg_full) if you need further inspection.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   unique_cmdb_count                                    sample_cmdb_ids
0                 29  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...,    unique_kpi_count                                   sample_kpi_names
0               361  [JVM-Memory_7778_JVM_Memory_HeapMemoryMax, JVM...,          cmdb_id                                           kpi_name  count            min            p50           mean            p95            max
0       Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31       0.000000       0.000000       0.000000       0.000000       0.000000
1       Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31     642.000000     642.000000     642.225806     643.000000     643.000000
2       Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  405396.000000  405396.000000  405396.000000  405396.000000  405396.000000
3   ServiceTest3                                            app.mrt     31       2.512830       3.915130       3.847425       4.862460       5.352320
4   ServiceTest3                                             app.rr     31      65.910000      75.880000      76.242258      82.995000      84.910000
..           ...                                                ...    ...            ...            ...            ...            ...            ...
45  ServiceTest8                                             app.sr     31      65.556522      73.918182      73.988993      80.426949      82.206186
46  ServiceTest9                                            app.cnt     31      54.000000     141.000000     161.935484     283.500000     361.000000
47  ServiceTest9                                            app.mrt     31       3.362590       3.910100       4.090768       5.129595       5.514150
48  ServiceTest9                                             app.rr     31      69.300000      75.270000      74.746129      78.870000      79.110000
49  ServiceTest5                                             app.sr     31      65.038462      74.635246      74.159920      79.256400      83.013072

[50 rows x 8 columns],          file                  earliest                    latest
0  metric.csv 2021-03-09 09:30:00+08:00 2021-03-09 10:00:00+08:00,    unique_cmdb_count                                    sample_cmdb_ids
0                 11  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat03, T...,    unique_trace_count                                 sample_trace_names
0                  66  [trace.self.duration_mean, trace.self.duration...,      cmdb_id                           kpi_name  count          min          p50         mean          p95          max
0   dockerB1            trace.self.duration_p95     31     0.002000     0.636200     0.514232     1.012000     1.013000
1   dockerB1           trace.self.duration_mean     31     0.002000     0.053340     0.051045     0.067680     0.086567
2   dockerB1               trace.self.row_count     31     1.000000  2138.000000  2268.387097  3927.000000  4565.000000
3       IG01        trace.to_Tomcat01.row_count     30   159.000000   229.500000   230.200000   282.850000   356.000000
4       IG01               trace.self.row_count     30  1390.000000  1820.000000  1831.500000  2200.600000  2700.000000
..       ...                                ...    ...          ...          ...          ...          ...          ...
45      MG02      trace.from_Tomcat04.row_count     30   182.000000   229.500000   229.300000   277.150000   323.000000
46      MG02  trace.from_dockerA1.duration_mean     30     0.006128     0.040113     0.048168     0.125950     0.189818
47      MG02   trace.from_dockerA1.duration_p95     30     0.007550     0.032575     0.249058     0.950517     1.013000
48      MG01  trace.from_dockerB2.duration_mean     30     0.007371     0.043731     0.047911     0.083843     0.104016
49      MG01   trace.from_dockerB2.duration_p95     30     0.008000     0.027625     0.270567     1.013175     1.015000

[50 rows x 8 columns],         file                  earliest                    latest
0  trace.csv 2021-03-09 09:30:00+08:00 2021-03-09 10:00:00+08:00,    unique_cmdb_count                                    sample_cmdb_ids
0                  7  [Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat...,    unique_log_count                                   sample_log_names
0                 3  [log.error_count, log.row_count, log.gc_pause_...,      cmdb_id          kpi_name  count    min      p50        mean        p95      max
0   Tomcat01     log.row_count     31   3.00  452.000  445.903226   557.5000   716.00
1   Tomcat01   log.error_count     31   0.00    0.000    0.000000     0.0000     0.00
2   apache02   log.error_count     31   0.00    0.000    0.838710     0.0000    26.00
3   apache01     log.row_count     31  13.00  918.000  889.806452  1058.5000  1365.00
4   Tomcat02   log.error_count     31   0.00    0.000    0.000000     0.0000     0.00
5   Tomcat02  log.gc_pause_max     31   1.87    1.930    2.011613     2.2800     2.93
6   Tomcat03   log.error_count     31   0.00    0.000    0.000000     0.0000     0.00
7   Tomcat02     log.row_count     31   1.00   33.000   32.064516    36.5000    39.00
8   Tomcat04   log.error_count     31   0.00    0.000    0.032258     0.0000     1.00
9   Tomcat03     log.row_count     31   8.00  458.000  445.677419   523.5000   688.00
10  apache02     log.row_count     31  14.00  920.000  893.645161  1074.0000  1363.00
11  apache01   log.error_count     31   0.00    0.000    0.612903     0.0000    19.00
12  Tomcat04     log.row_count     31   9.00  468.000  447.064516   551.0000   666.00
13  Tomcat04  log.gc_pause_max     24   0.02    0.075    0.065833     0.1085     0.13
14  Tomcat01  log.gc_pause_max     23   0.02    0.070    0.062174     0.0990     0.10
15   Mysql01   log.error_count     22   0.00    0.000    0.000000     0.0000     0.00
16   Mysql01     log.row_count     22   1.00    3.000    3.636364     7.9000    10.00
17  Tomcat03  log.gc_pause_max     22   0.02    0.060    0.054091     0.0800     0.08,       file                  earliest                    latest
0  log.csv 2021-03-09 09:30:00+08:00 2021-03-09 10:00:00+08:00, 46,     cmdb_id  count
0  apache02     26
1  apache01     19
2  Tomcat04      1,     cmdb_id                                    sample_messages
0  apache02  [IPAddress "POST /UOCP/base/ServiceTest7.json ...
1  apache01  [IPAddress "POST /UOCP/base/ServiceTest2.json ...
2  Tomcat04  [IG02 POST /UOCP/base/ServiceTest4.json HTTP/1...,              file                  earliest                    latest
0  error_logs.csv 2021-03-09 09:53:26+08:00 2021-03-09 09:53:42+08:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df from kernel if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timestamp and dt with UTC->Asia/Shanghai tz
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# Ensure value numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global thresholds using the full series (metric_thresholds)
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        full_count='count',
        global_min='min',
        global_p50=lambda x: x.quantile(0.5),
        global_mean='mean',
        global_p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
)

# 2) Filter metric.csv to the incident time window (inclusive)
start_ts = pd.to_datetime('2021-03-09 09:30:00').tz_localize(tz)
end_ts   = pd.to_datetime('2021-03-09 10:00:00').tz_localize(tz)

metric_incident = metric_df.loc[(metric_df['dt'] >= start_ts) & (metric_df['dt'] <= end_ts)].copy()

# 3) Mark anomalous points in the incident window when value >= global p95
# Join metric_incident with metric_thresholds to get global_p95 per series
metric_incident_with_thresh = metric_incident.merge(
    metric_thresholds[['cmdb_id', 'kpi_name', 'global_p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Flag anomalies
metric_incident_with_thresh['is_anomaly'] = metric_incident_with_thresh['value'] >= metric_incident_with_thresh['global_p95']

# Aggregate anomalies per (cmdb_id, kpi_name)
agg = (
    metric_incident_with_thresh.loc[metric_incident_with_thresh['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('dt', 'min'),
        max_anomaly_value=('value', 'max')
    )
    .reset_index()
)

# Need full_count_in_window for each pair (count of points in the incident window)
full_count_in_window = (
    metric_incident
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .count()
    .reset_index(name='full_count_in_window')
)

# Join agg with global p95 and full_count_in_window
metric_anomalies = (
    agg
    .merge(metric_thresholds[['cmdb_id', 'kpi_name', 'global_p95']], on=['cmdb_id', 'kpi_name'], how='left')
    .merge(full_count_in_window, on=['cmdb_id', 'kpi_name'], how='left')
)

# If there are KPI pairs with zero anomalies they won't appear in metric_anomalies (as required)
# Sort by anomaly_count desc and limit to top 20 for display
metric_anomalies = metric_anomalies.sort_values('anomaly_count', ascending=False).reset_index(drop=True)

# 4) Compute earliest anomaly timestamp across all anomalies and total number anomaly points within window
if not metric_anomalies.empty:
    earliest_anomaly_all = metric_anomalies['earliest_anomaly_timestamp'].min()
    # total anomaly points is sum of anomaly_count
    total_anomalies_count = int(metric_anomalies['anomaly_count'].sum())
else:
    earliest_anomaly_all = pd.NaT
    total_anomalies_count = 0

# Keep full intermediate tables in kernel variables as required:
# metric_thresholds, metric_incident, metric_anomalies already set

# Final compact outputs: top 20 rows of metric_anomalies, earliest anomaly timestamp, total anomaly points
metric_anomalies.head(20), earliest_anomaly_all, total_anomalies_count

```
Out[2]:
```
```python
# Summarize the anomaly analysis results for metric.csv in plain English as a single string.
# Reuse existing kernel variables: metric_thresholds, metric_incident, metric_anomalies

n_series_with_anomalies = int(metric_anomalies.shape[0])
total_anomaly_points = int(total_anomalies_count) if 'total_anomalies_count' in globals() else int(metric_anomalies['anomaly_count'].sum())
earliest_anomaly = metric_anomalies['earliest_anomaly_timestamp'].min() if n_series_with_anomalies > 0 else pd.NaT

# Prepare a compact top-5 sample lines for the user (component | kpi_name | anomaly_count | global_p95 | max_anomaly_value)
top5 = metric_anomalies.head(5)[['cmdb_id','kpi_name','anomaly_count','global_p95','max_anomaly_value']].copy()
top5_lines = []
for _, r in top5.iterrows():
    top5_lines.append(f"{r['cmdb_id']} | {r['kpi_name'][:80]}{'...' if len(r['kpi_name'])>80 else ''} | count={int(r['anomaly_count'])} | p95={r['global_p95']:.6g} | max={r['max_anomaly_value']:.6g}")

summary = (
    "Metric analysis summary (metric.csv):\n\n"
    f"- Saved kernel variables: metric_thresholds (global stats per series), metric_incident (data in 2021-03-09 09:30:00+08 through 2021-03-09 10:00:00+08), "
    "metric_anomalies (aggregated anomalies in that window).\n\n"
    f"- Number of (cmdb_id, kpi_name) series with >=1 anomaly in the window: {n_series_with_anomalies}\n"
    f"- Total anomaly points within the window: {total_anomaly_points}\n"
    f"- Earliest anomaly timestamp (UTC+8): {earliest_anomaly}\n\n"
    "- Detection rule: point is anomalous if value >= the series' global p95 (computed from the full available series).\n\n"
    "- Compact top-5 series by anomaly_count (component | kpi_name (truncated) | anomaly_count | global_p95 | max_anomaly_value):\n"
    + "\n".join("  - " + s for s in top5_lines) + "\n\n"
    "Observation: Many top series have anomaly_count equal to the full window length (31), meaning every point in the incident window exceeded that series' global p95.\n"
    "If you want the full metric_anomalies table or specific series details, the full tables are stored in the kernel variables named above."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  anomaly_count earliest_anomaly_timestamp  max_anomaly_value  global_p95  full_count_in_window
0   apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31  2021-03-09 09:30:00+08:00                4.0         4.0                    31
1   apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
2   Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
3   Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
4    Redis02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
5   Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31  2021-03-09 09:30:00+08:00              100.0       100.0                    31
6    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
7   Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31  2021-03-09 09:30:00+08:00              500.0       500.0                    31
8   Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
9       IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-09 09:30:00+08:00                0.3         0.3                    31
10  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
11      IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31  2021-03-09 09:30:00+08:00                3.0         3.0                    31
12  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
13  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
14  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31  2021-03-09 09:30:00+08:00           405396.0    405396.0                    31
15   Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
16      IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31  2021-03-09 09:30:00+08:00                3.0         3.0                    31
17  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
18  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31  2021-03-09 09:30:00+08:00                0.0         0.0                    31
19  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31  2021-03-09 09:30:00+08:00           368554.0    368554.0                    31, Timestamp('2021-03-09 09:30:00+0800', tz='Asia/Shanghai'), 14398)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure timestamp numeric and timezone-aware datetime column (UTC -> Asia/Shanghai)
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)

# Ensure value numeric
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global thresholds on full series (trace_thresholds)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(
        full_count='count',
        global_min='min',
        global_p5=lambda x: x.quantile(0.05),
        global_p50=lambda x: x.quantile(0.50),
        global_mean='mean',
        global_p95=lambda x: x.quantile(0.95),
        global_max='max'
    )
    .reset_index()
)

# 3) Classify each trace_name as 'traffic' or 'latency' (case-insensitive)
def classify_trace(name: str) -> str:
    if not isinstance(name, str):
        return 'other'
    n = name.lower()
    if ('row_count' in n) or ('to_' in n) or ('from_' in n):
        return 'traffic'
    if ('duration' in n) or ('p95' in n) or ('mean' in n):
        return 'latency'
    return 'other'

trace_thresholds['type'] = trace_thresholds['trace_name'].apply(classify_trace)

# 2) Filter trace.csv to incident window and save
start_ts = pd.to_datetime('2021-03-09 09:30:00').tz_localize(tz)
end_ts   = pd.to_datetime('2021-03-09 10:00:00').tz_localize(tz)
trace_incident = trace_df.loc[(trace_df['dt'] >= start_ts) & (trace_df['dt'] <= end_ts)].copy()

# Prepare a small thresholds slice to join into incident rows
thresholds_slice = trace_thresholds[['cmdb_id', 'trace_name', 'global_p5', 'global_p95', 'type']]

# 4) Join thresholds into incident and mark anomalies per rules
trace_incident_with_thresh = trace_incident.merge(
    thresholds_slice,
    on=['cmdb_id', 'trace_name'],
    how='left'
)

# Flag anomalies: traffic -> value <= global_p5 ; latency -> value >= global_p95. Others: no anomaly.
trace_incident_with_thresh['is_anomaly'] = False
mask_traffic = trace_incident_with_thresh['type'] == 'traffic'
mask_latency = trace_incident_with_thresh['type'] == 'latency'

trace_incident_with_thresh.loc[mask_traffic, 'is_anomaly'] = (
    trace_incident_with_thresh.loc[mask_traffic, 'value'] <= trace_incident_with_thresh.loc[mask_traffic, 'global_p5']
)
trace_incident_with_thresh.loc[mask_latency, 'is_anomaly'] = (
    trace_incident_with_thresh.loc[mask_latency, 'value'] >= trace_incident_with_thresh.loc[mask_latency, 'global_p95']
)

# 5) Aggregate anomalies per (cmdb_id, trace_name)
# Compute full_count_in_window
full_count_in_window = (
    trace_incident
    .groupby(['cmdb_id', 'trace_name'])['value']
    .count()
    .reset_index(name='full_count_in_window')
)

# Split anomalies by type to compute extremal values appropriately
anoms = trace_incident_with_thresh.loc[trace_incident_with_thresh['is_anomaly']].copy()

# Traffic anomalies: extremal_anomaly_value = min(value), threshold used = global_p5
traffic_anoms = anoms[anoms['type'] == 'traffic']
if not traffic_anoms.empty:
    traffic_agg = traffic_anoms.groupby(['cmdb_id', 'trace_name', 'type']).agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('dt', 'min'),
        extremal_anomaly_value=('value', 'min')
    ).reset_index()
    traffic_agg['global_threshold_used'] = traffic_agg.merge(
        trace_thresholds[['cmdb_id','trace_name','global_p5']],
        on=['cmdb_id','trace_name'],
        how='left'
    )['global_p5'].values
else:
    traffic_agg = pd.DataFrame(columns=['cmdb_id','trace_name','type','anomaly_count','earliest_anomaly_timestamp','extremal_anomaly_value','global_threshold_used'])

# Latency anomalies: extremal_anomaly_value = max(value), threshold used = global_p95
latency_anoms = anoms[anoms['type'] == 'latency']
if not latency_anoms.empty:
    latency_agg = latency_anoms.groupby(['cmdb_id', 'trace_name', 'type']).agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('dt', 'min'),
        extremal_anomaly_value=('value', 'max')
    ).reset_index()
    latency_agg['global_threshold_used'] = latency_agg.merge(
        trace_thresholds[['cmdb_id','trace_name','global_p95']],
        on=['cmdb_id','trace_name'],
        how='left'
    )['global_p95'].values
else:
    latency_agg = pd.DataFrame(columns=['cmdb_id','trace_name','type','anomaly_count','earliest_anomaly_timestamp','extremal_anomaly_value','global_threshold_used'])

# Combine aggregated anomalies
trace_anomalies = pd.concat([traffic_agg, latency_agg], ignore_index=True)

# Join full_count_in_window
if not trace_anomalies.empty:
    trace_anomalies = trace_anomalies.merge(full_count_in_window, on=['cmdb_id','trace_name'], how='left')
else:
    trace_anomalies['full_count_in_window'] = pd.Series(dtype='int')

# Reorder/select columns as requested and sort by anomaly_count desc
cols = ['cmdb_id', 'trace_name', 'type', 'anomaly_count', 'earliest_anomaly_timestamp',
        'extremal_anomaly_value', 'global_threshold_used', 'full_count_in_window']
# Ensure columns exist
for c in cols:
    if c not in trace_anomalies.columns:
        trace_anomalies[c] = pd.NA

trace_anomalies = trace_anomalies[cols].sort_values('anomaly_count', ascending=False).reset_index(drop=True)

# 6) Top 20 aggregated trace anomaly rows, earliest anomaly timestamp across all, and total number of anomaly points
trace_anomalies_top20 = trace_anomalies.head(20)

if not trace_anomalies.empty:
    earliest_anomaly_all = trace_anomalies['earliest_anomaly_timestamp'].min()
    total_trace_anomalies = int(trace_anomalies['anomaly_count'].sum())
else:
    earliest_anomaly_all = pd.NaT
    total_trace_anomalies = 0

# Keep full intermediate tables in kernel variables: trace_thresholds, trace_incident, trace_anomalies
trace_thresholds, trace_incident, trace_anomalies_top20, earliest_anomaly_all, total_trace_anomalies

```
Out[3]:
```
```python
# Compose a plain-English summary of the trace anomaly detection results using existing kernel variables.
# This cell reuses trace_thresholds, trace_incident, trace_anomalies, earliest_anomaly_all, total_trace_anomalies.

import pandas as pd

# Safely access variables (they should exist from previous steps)
ta = trace_anomalies if 'trace_anomalies' in globals() else pd.DataFrame()
tt = trace_thresholds if 'trace_thresholds' in globals() else pd.DataFrame()
ti = trace_incident if 'trace_incident' in globals() else pd.DataFrame()
earliest = earliest_anomaly_all if 'earliest_anomaly_all' in globals() else pd.NaT
total_count = int(total_trace_anomalies) if 'total_trace_anomalies' in globals() else (int(ta['anomaly_count'].sum()) if not ta.empty else 0)

n_series = int(ta.shape[0])

# Prepare top-5 compact lines
top5 = []
if not ta.empty:
    sample = ta.head(5)[['cmdb_id','trace_name','type','anomaly_count','global_threshold_used','extremal_anomaly_value']].copy()
    for _, r in sample.iterrows():
        name = (r['trace_name'][:80] + '...') if len(r['trace_name'])>80 else r['trace_name']
        top5.append(f"{r['cmdb_id']} | {name} | type={r['type']} | anomalies={int(r['anomaly_count'])} | threshold={r['global_threshold_used']:.6g} | extremal={r['extremal_anomaly_value']:.6g}")

summary = (
    "Trace anomaly detection summary (trace.csv):\n\n"
    "- Saved kernel variables: trace_thresholds (global stats per series), trace_incident (data in the incident window),\n"
    "  trace_anomalies (aggregated anomalies per series).\n\n"
    f"- Detection rules used: 'traffic' series flagged when value <= series' global p5 (traffic drop); "
    f"'latency' series flagged when value >= series' global p95 (latency spike).\n\n"
    f"- Number of (cmdb_id, trace_name) series with >=1 anomaly in the window: {n_series}\n"
    f"- Total anomaly points within the window: {total_count}\n"
    f"- Earliest anomaly timestamp across all trace anomalies (UTC+8): {earliest}\n\n"
    "- Top-5 anomalous series (compact):\n"
    + ("\n".join("  - " + s for s in top5) if top5 else "  - (no anomalies detected)") + "\n\n"
    "Notes:\n"
    "- Full per-series thresholds are in trace_thresholds; incident window rows are in trace_incident; aggregated anomalies are in trace_anomalies.\n"
    "- The analysis used the full historical series to compute p5/p95 before filtering to the incident window."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                     trace_name  full_count  global_min  global_p5  global_p50  global_mean  global_p95  global_max     type
0        IG01  trace.from_root.duration_mean           4    0.385700   0.480845    1.786000     1.887508    3.436283    3.592333  traffic
1        IG01   trace.from_root.duration_p95           4    1.020000   1.067010    6.296100     7.296050   14.925020   15.572000  traffic
2        IG01      trace.from_root.row_count           4    1.000000   1.900000    8.000000     6.750000    9.850000   10.000000  traffic
3        IG01       trace.self.duration_mean          30    3.711414   3.795632    4.127564     4.156079    4.614003    4.804995  latency
4        IG01        trace.self.duration_p95          30   15.530950  15.537912   15.556000    15.557170   15.579100   15.584000  latency
..        ...                            ...         ...         ...        ...         ...          ...         ...         ...      ...
229  dockerB2     trace.to_MG01.duration_p95          30    0.008000   0.008000    0.027625     0.270567    1.013175    1.015000  traffic
230  dockerB2        trace.to_MG01.row_count          30   55.000000  57.800000  138.000000   140.500000  237.650000  270.000000  traffic
231  dockerB2    trace.to_MG02.duration_mean          30    0.016366   0.022491    0.048692     0.048824    0.072973    0.094149  traffic
232  dockerB2     trace.to_MG02.duration_p95          30    0.008000   0.008830    0.053550     0.211653    0.957325    1.013000  traffic
233  dockerB2        trace.to_MG02.row_count          30   49.000000  58.900000  140.500000   139.233333  240.550000  251.000000  traffic

[234 rows x 10 columns],        timestamp   cmdb_id                       trace_name        value                        dt
0     1615253400      IG01         trace.self.duration_mean     4.239813 2021-03-09 09:30:00+08:00
1     1615253400      IG01          trace.self.duration_p95    15.578000 2021-03-09 09:30:00+08:00
2     1615253400      IG01             trace.self.row_count  1496.000000 2021-03-09 09:30:00+08:00
3     1615253400      IG01  trace.to_Tomcat01.duration_mean     0.213483 2021-03-09 09:30:00+08:00
4     1615253400      IG01   trace.to_Tomcat01.duration_p95     1.068000 2021-03-09 09:30:00+08:00
...          ...       ...                              ...          ...                       ...
6178  1615255140  dockerB2       trace.to_MG02.duration_p95     0.012850 2021-03-09 09:59:00+08:00
6179  1615255140  dockerB2          trace.to_MG02.row_count   142.000000 2021-03-09 09:59:00+08:00
6180  1615255200  dockerB1         trace.self.duration_mean     0.002000 2021-03-09 10:00:00+08:00
6181  1615255200  dockerB1          trace.self.duration_p95     0.002000 2021-03-09 10:00:00+08:00
6182  1615255200  dockerB1             trace.self.row_count     1.000000 2021-03-09 10:00:00+08:00

[6183 rows x 5 columns],      cmdb_id                         trace_name     type  anomaly_count earliest_anomaly_timestamp  extremal_anomaly_value  global_threshold_used  full_count_in_window
0   dockerB1            trace.self.duration_p95  latency              7  2021-03-09 09:36:00+08:00                1.013000               1.012000                    31
1   dockerA2            trace.self.duration_p95  latency              6  2021-03-09 09:37:00+08:00                1.018000               1.017000                    30
2   dockerA2         trace.to_MG02.duration_p95  traffic              5  2021-03-09 09:30:00+08:00                0.007900               0.008000                    30
3   dockerB2         trace.to_MG01.duration_p95  traffic              5  2021-03-09 09:30:00+08:00                0.008000               0.008000                    30
4       MG02   trace.from_dockerA2.duration_p95  traffic              5  2021-03-09 09:30:00+08:00                0.007900               0.008000                    30
5       MG01   trace.from_dockerB2.duration_p95  traffic              5  2021-03-09 09:30:00+08:00                0.008000               0.008000                    30
6       MG01   trace.from_dockerA2.duration_p95  traffic              4  2021-03-09 09:42:00+08:00                0.007650               0.008000                    30
7   dockerA2         trace.to_MG01.duration_p95  traffic              4  2021-03-09 09:42:00+08:00                0.007650               0.008000                    30
8   Tomcat01            trace.self.duration_p95  latency              4  2021-03-09 09:49:00+08:00                1.033000               1.004000                    30
9       MG02      trace.from_dockerA1.row_count  traffic              3  2021-03-09 09:53:00+08:00                5.000000               7.000000                    30
10  dockerA1            trace.to_MG02.row_count  traffic              3  2021-03-09 09:53:00+08:00                5.000000               7.000000                    30
11  dockerA1            trace.to_MG01.row_count  traffic              3  2021-03-09 09:52:00+08:00                3.000000               7.000000                    30
12      MG02      trace.from_Tomcat04.row_count  traffic              3  2021-03-09 09:33:00+08:00              182.000000             185.000000                    30
13      MG01      trace.from_dockerA1.row_count  traffic              3  2021-03-09 09:52:00+08:00                3.000000               7.000000                    30
14  Tomcat04            trace.to_MG02.row_count  traffic              3  2021-03-09 09:33:00+08:00              182.000000             185.000000                    30
15      IG02               trace.self.row_count  traffic              2  2021-03-09 09:34:00+08:00             1332.000000            1511.400000                    30
16      IG01        trace.to_Tomcat04.row_count  traffic              2  2021-03-09 09:34:00+08:00              176.000000             181.900000                    30
17      IG02    trace.to_Tomcat03.duration_mean  traffic              2  2021-03-09 09:30:00+08:00                0.233133               0.244835                    30
18      MG01   trace.from_Tomcat04.duration_p95  traffic              2  2021-03-09 09:30:00+08:00                1.050000               1.061295                    30
19      MG01  trace.from_Tomcat04.duration_mean  traffic              2  2021-03-09 09:30:00+08:00                0.129893               0.187714                    30, Timestamp('2021-03-09 09:30:00+0800', tz='Asia/Shanghai'), 471)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes if present, else load
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

# Normalize timestamps and timezone (UTC -> Asia/Shanghai) and ensure numeric values
for df in (log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
        df['dt'] = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['dt'] = pd.NaT

# Ensure numeric value for log_df
log_df['value'] = pd.to_numeric(log_df.get('value', pd.Series(dtype='float')), errors='coerce')

# Ensure there is a 'log_name' column (some prior steps may have renamed it)
if 'log_name' not in log_df.columns:
    if 'kpi_name' in log_df.columns:
        log_df = log_df.rename(columns={'kpi_name': 'log_name'})
    elif 'log' in log_df.columns:
        log_df = log_df.rename(columns={'log': 'log_name'})
    else:
        # If no name-like column, create empty to avoid crashes
        log_df['log_name'] = log_df.get('log_name', pd.NA)

# Incident window
start_ts = pd.to_datetime('2021-03-09 09:30:00').tz_localize(tz)
end_ts   = pd.to_datetime('2021-03-09 10:00:00').tz_localize(tz)

# 1) Compute global statistics for each (cmdb_id, log_name) over full series -> log_thresholds
log_thresholds = (
    log_df
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(
        full_count='count',
        global_min='min',
        global_p5=lambda x: x.quantile(0.05),
        global_p50=lambda x: x.quantile(0.50),
        global_mean='mean',
        global_p95=lambda x: x.quantile(0.95),
        global_max='max'
    )
    .reset_index()
)

# 2) Filter log.csv to the incident window -> log_incident
log_incident = log_df.loc[(log_df['dt'] >= start_ts) & (log_df['dt'] <= end_ts)].copy()

# 3) Classify log_name types
def classify_log(name: str) -> str:
    if not isinstance(name, str):
        return 'other'
    n = name.lower()
    if 'error' in n:
        return 'errors'
    if 'row_count' in n:
        return 'traffic'
    if 'gc_pause' in n or 'gc' in n:
        return 'latency'
    return 'other'

log_thresholds['type'] = log_thresholds['log_name'].apply(classify_log)

# Prepare thresholds slice for join
thresholds_slice = log_thresholds[['cmdb_id', 'log_name', 'global_p5', 'global_p95', 'type']]

# 4) Flag anomalies in log_incident according to rules using global thresholds
log_incident_with_thresh = log_incident.merge(thresholds_slice, on=['cmdb_id', 'log_name'], how='left')

# Initialize flag
log_incident_with_thresh['is_anomaly'] = False

# errors: high -> value >= global_p95
mask_errors = log_incident_with_thresh['type'] == 'errors'
log_incident_with_thresh.loc[mask_errors, 'is_anomaly'] = (
    log_incident_with_thresh.loc[mask_errors, 'value'] >= log_incident_with_thresh.loc[mask_errors, 'global_p95']
)

# traffic: low -> value <= global_p5
mask_traffic = log_incident_with_thresh['type'] == 'traffic'
log_incident_with_thresh.loc[mask_traffic, 'is_anomaly'] = (
    log_incident_with_thresh.loc[mask_traffic, 'value'] <= log_incident_with_thresh.loc[mask_traffic, 'global_p5']
)

# latency: high -> value >= global_p95
mask_latency = log_incident_with_thresh['type'] == 'latency'
log_incident_with_thresh.loc[mask_latency, 'is_anomaly'] = (
    log_incident_with_thresh.loc[mask_latency, 'value'] >= log_incident_with_thresh.loc[mask_latency, 'global_p95']
)

# 5) Aggregate flagged log anomalies per (cmdb_id, log_name)
# Compute full_count_in_window
full_count_in_window = (
    log_incident
    .groupby(['cmdb_id', 'log_name'])['value']
    .count()
    .reset_index(name='full_count_in_window')
)

# Filter anomalies
log_anoms = log_incident_with_thresh.loc[log_incident_with_thresh['is_anomaly']].copy()

if not log_anoms.empty:
    # Group by cmdb_id, log_name, type to compute counts and extremal values
    def compute_group(g):
        typ = g['type'].iloc[0]
        anomaly_count = int(g['is_anomaly'].sum())
        earliest = g['dt'].min()
        if typ == 'traffic':
            extremal = float(g['value'].min())
        else:
            extremal = float(g['value'].max())
        return pd.Series({
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest,
            'extremal_anomaly_value': extremal
        })
    grouped = log_anoms.groupby(['cmdb_id', 'log_name', 'type']).apply(compute_group).reset_index()

    # Attach thresholds and choose global_threshold_used
    grouped = grouped.merge(log_thresholds[['cmdb_id','log_name','global_p5','global_p95','type']],
                            on=['cmdb_id','log_name','type'], how='left')
    grouped['global_threshold_used'] = grouped.apply(
        lambda r: r['global_p95'] if r['type'] in ('errors','latency') else r['global_p5'],
        axis=1
    )
    # Join full_count_in_window
    log_anomalies = grouped.merge(full_count_in_window, on=['cmdb_id','log_name'], how='left')
    # Select and reorder columns
    log_anomalies = log_anomalies[['cmdb_id','log_name','type','anomaly_count','earliest_anomaly_timestamp',
                                   'extremal_anomaly_value','global_threshold_used','full_count_in_window']]
    log_anomalies = log_anomalies.sort_values('anomaly_count', ascending=False).reset_index(drop=True)
else:
    log_anomalies = pd.DataFrame(columns=['cmdb_id','log_name','type','anomaly_count','earliest_anomaly_timestamp',
                                          'extremal_anomaly_value','global_threshold_used','full_count_in_window'])

# Ensure variables saved in kernel: log_thresholds, log_incident, log_anomalies
# 6) error_logs.csv: filter to incident window and compute requested summaries
filtered_error_logs = error_df.loc[(error_df['dt'] >= start_ts) & (error_df['dt'] <= end_ts)].copy()
filtered_error_count = len(filtered_error_logs)

# Counts per cmdb_id top 20
filtered_error_counts_top20 = filtered_error_logs.groupby('cmdb_id').size().sort_values(ascending=False).head(20).reset_index(name='count')

# Earliest and latest error timestamp
if filtered_error_count > 0:
    filtered_error_earliest = filtered_error_logs['dt'].min()
    filtered_error_latest = filtered_error_logs['dt'].max()
else:
    filtered_error_earliest = pd.NaT
    filtered_error_latest = pd.NaT

# Up to 5 sample messages per top-3 cmdb_id by count
top3_cmdb = list(filtered_error_counts_top20['cmdb_id'].head(3))
def sample_messages(cid):
    msgs = filtered_error_logs.loc[filtered_error_logs['cmdb_id']==cid, 'message'].dropna().astype(str).unique()[:5]
    return list(msgs)

error_samples_top3 = pd.DataFrame({
    'cmdb_id': top3_cmdb,
    'sample_messages': [sample_messages(cid) for cid in top3_cmdb]
})

# Prepare a small range summary dataframe
filtered_error_range = pd.DataFrame({
    'total_rows': [filtered_error_count],
    'earliest': [filtered_error_earliest],
    'latest': [filtered_error_latest]
})

# Final compact outputs: top 20 log anomalies, and filtered error logs summary (counts + samples)
log_anomalies_top20 = log_anomalies.head(20)

log_anomalies_top20, filtered_error_counts_top20, error_samples_top3, filtered_error_range

```
Out[4]:
```
```python
# Summarize log and error_logs analysis results in plain English.
# Reuse kernel variables: log_anomalies_top20, filtered_error_counts_top20, error_samples_top3, filtered_error_range

# Build compact top-lines for log anomalies (up to 8 lines)
lines = []
if 'log_anomalies_top20' in globals() and not log_anomalies_top20.empty:
    sample = log_anomalies_top20.head(8)
    for _, r in sample.iterrows():
        lines.append(f"{r['cmdb_id']} | {r['log_name']} | type={r['type']} | anomalies={int(r['anomaly_count'])} | earliest={r['earliest_anomaly_timestamp']} | extremal={r['extremal_anomaly_value']:.6g} | threshold={r['global_threshold_used']:.6g} | window_count={int(r['full_count_in_window'])}")
else:
    lines.append("No log anomalies detected.")

# Error logs compact summary
if 'filtered_error_range' in globals() and not filtered_error_range.empty:
    total_errors = int(filtered_error_range.loc[0, 'total_rows'])
    err_earliest = filtered_error_range.loc[0, 'earliest']
    err_latest = filtered_error_range.loc[0, 'latest']
else:
    total_errors = 0
    err_earliest = pd.NaT
    err_latest = pd.NaT

# Top counts for error logs (up to top 3 shown)
top_err_counts = []
if 'filtered_error_counts_top20' in globals() and not filtered_error_counts_top20.empty:
    for _, r in filtered_error_counts_top20.head(3).iterrows():
        top_err_counts.append((r['cmdb_id'], int(r['count'])))
else:
    top_err_counts = []

# Sample messages for top-3
samples_lines = []
if 'error_samples_top3' in globals() and not error_samples_top3.empty:
    for _, r in error_samples_top3.iterrows():
        samples_lines.append(f"{r['cmdb_id']}: {r['sample_messages']}")
else:
    samples_lines.append("No sample error messages.")

summary = (
    "Summary of log and error_logs analysis (incident window 2021-03-09 09:30–10:00 UTC+8):\n\n"
    "1) Log anomalies (top aggregated results):\n"
    f"- {len(log_anomalies_top20)} aggregated series returned (top 20 shown).\n"
    "- Representative top anomalies:\n"
    + "\n".join("  - " + s for s in lines) + "\n\n"
    "Key observations from logs:\n"
    "- Multiple components have sustained 'log.error_count' anomalies covering the full window (anomaly_count=31), including Tomcat02, Tomcat01, apache01, apache02, Tomcat03, Tomcat04. "
    "Note: these series' global p95 thresholds are 0.0 in many cases, so non-negative error_count values were flagged under the >= p95 rule.\n"
    "- Traffic drops detected (log.row_count) for some Tomcat and Mysql series (e.g., Tomcat02 had 15 anomaly points; earliest ~09:31).\n"
    "- GC pause (latency) spikes detected in a few Tomcat nodes (e.g., Tomcat02, Tomcat03, Tomcat01) with small counts but elevated pause durations.\n\n"
    "2) error_logs (filtered to incident window):\n"
    f"- Total error log rows in window: {total_errors}\n"
    f"- Top components by error-log count (top 3): {top_err_counts}\n"
    f"- Earliest error timestamp in window: {err_earliest}\n"
    f"- Latest error timestamp in window: {err_latest}\n"
    "- Sample messages for top-3 components:\n"
    + "\n".join("  - " + s for s in samples_lines) + "\n\n"
    "Saved kernel variables for follow-up:\n"
    "- log_thresholds (global per-series stats), log_incident (rows in window), log_anomalies (aggregated anomalies), filtered_error_logs (error rows in window).\n\n"
    "Notes:\n"
    "- Detection rules used: errors -> high (>= global p95), traffic -> low (<= global p5), gc/latency -> high (>= global p95).\n"
    "- Many 'log.error_count' series had global p95 = 0.0, which caused all non-negative points in the incident window to be marked as anomalies; review thresholds if this is unexpected.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name     type  anomaly_count earliest_anomaly_timestamp  extremal_anomaly_value  global_threshold_used  full_count_in_window
0   Tomcat02   log.error_count   errors             31  2021-03-09 09:30:00+08:00                    0.00                 0.0000                    31
1   Tomcat01   log.error_count   errors             31  2021-03-09 09:30:00+08:00                    0.00                 0.0000                    31
2   apache01   log.error_count   errors             31  2021-03-09 09:30:00+08:00                   19.00                 0.0000                    31
3   apache02   log.error_count   errors             31  2021-03-09 09:30:00+08:00                   26.00                 0.0000                    31
4   Tomcat03   log.error_count   errors             31  2021-03-09 09:30:00+08:00                    0.00                 0.0000                    31
5   Tomcat04   log.error_count   errors             31  2021-03-09 09:30:00+08:00                    1.00                 0.0000                    31
6    Mysql01   log.error_count   errors             22  2021-03-09 09:30:00+08:00                    0.00                 0.0000                    22
7   Tomcat02     log.row_count  traffic             15  2021-03-09 09:31:00+08:00                    1.00                32.0000                    31
8   Tomcat02  log.gc_pause_max  latency              4  2021-03-09 09:32:00+08:00                    2.93                 2.2800                    31
9    Mysql01     log.row_count  traffic              3  2021-03-09 09:38:00+08:00                    1.00                 1.0000                    22
10  Tomcat03  log.gc_pause_max  latency              3  2021-03-09 09:33:00+08:00                    0.08                 0.0800                    22
11  Tomcat01     log.row_count  traffic              2  2021-03-09 09:53:00+08:00                    3.00               366.0000                    31
12  Tomcat01  log.gc_pause_max  latency              2  2021-03-09 09:52:00+08:00                    0.10                 0.0990                    23
13  Tomcat03     log.row_count  traffic              2  2021-03-09 09:30:00+08:00                    8.00               365.5000                    31
14  Tomcat04     log.row_count  traffic              2  2021-03-09 09:34:00+08:00                    9.00               361.5000                    31
15  Tomcat04  log.gc_pause_max  latency              2  2021-03-09 09:30:00+08:00                    0.13                 0.1085                    24
16  apache01     log.row_count  traffic              2  2021-03-09 09:34:00+08:00                   13.00               748.0000                    31
17  apache02     log.row_count  traffic              2  2021-03-09 09:34:00+08:00                   14.00               759.5000                    31,     cmdb_id  count
0  apache02     26
1  apache01     19
2  Tomcat04      1,     cmdb_id                                    sample_messages
0  apache02  [IPAddress "POST /UOCP/base/ServiceTest7.json ...
1  apache01  [IPAddress "POST /UOCP/base/ServiceTest2.json ...
2  Tomcat04  [IG02 POST /UOCP/base/ServiceTest4.json HTTP/1...,    total_rows                  earliest                    latest
0          46 2021-03-09 09:53:26+08:00 2021-03-09 09:53:42+08:00)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: trace_incident, trace_thresholds, filtered_error_logs
# If missing, attempt to load minimally (defensive)
try:
    trace_incident
except NameError:
    trace_incident = pd.read_csv('trace.csv')
    trace_incident['timestamp'] = pd.to_numeric(trace_incident['timestamp'], errors='coerce').astype('Int64')
    trace_incident['dt'] = pd.to_datetime(trace_incident['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)
    trace_incident['value'] = pd.to_numeric(trace_incident['value'], errors='coerce')

try:
    trace_thresholds
except NameError:
    tt = trace_incident.groupby(['cmdb_id','trace_name'])['value'].agg(
        full_count='count',
        global_p5=lambda x: x.quantile(0.05),
        global_p95=lambda x: x.quantile(0.95)
    ).reset_index()
    trace_thresholds = tt

try:
    filtered_error_logs
except NameError:
    # load error_logs.csv and create filtered_error_logs for full incident window if absent
    error_df = pd.read_csv('error_logs.csv')
    error_df['timestamp'] = pd.to_numeric(error_df['timestamp'], errors='coerce').astype('Int64')
    error_df['dt'] = pd.to_datetime(error_df['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz)
    # filtered_error_logs earlier was for 09:30-10:00; recreate for safety
    start_full = pd.to_datetime('2021-03-09 09:30:00').tz_localize(tz)
    end_full   = pd.to_datetime('2021-03-09 10:00:00').tz_localize(tz)
    filtered_error_logs = error_df.loc[(error_df['dt'] >= start_full) & (error_df['dt'] <= end_full)].copy()

# 1) Filter trace_incident to 09:52:00 - 09:54:00 inclusive and selected cmdb_ids
start_window = pd.to_datetime('2021-03-09 09:52:00').tz_localize(tz)
end_window   = pd.to_datetime('2021-03-09 09:54:00').tz_localize(tz)
focus_cmdbs = ['apache01','apache02','MG01','MG02','dockerA1','dockerA2','IG01',
               'Tomcat01','Tomcat02','Tomcat03','Tomcat04']

trace_focus = trace_incident.loc[
    (trace_incident['dt'] >= start_window) &
    (trace_incident['dt'] <= end_window) &
    (trace_incident['cmdb_id'].isin(focus_cmdbs))
].copy()

# 2) For each (cmdb_id, trace_name) in trace_focus compute required summary
if not trace_focus.empty:
    # ensure thresholds accessible: trace_thresholds has global_p5 and global_p95
    thresholds = trace_thresholds[['cmdb_id','trace_name','global_p5','global_p95']].drop_duplicates()

    # classify by name for this task
    def classify(name: str) -> str:
        if not isinstance(name, str):
            return 'other'
        n = name.lower()
        if 'row_count' in n:
            return 'traffic'
        if ('duration' in n) or ('p95' in n) or ('mean' in n):
            return 'latency'
        return 'other'

    # compute window_count, min/max and earliest breach/min timestamp per series
    grouped = trace_focus.groupby(['cmdb_id','trace_name'])
    rows = []
    for (cid, tname), g in grouped:
        typ = classify(tname)
        window_count = int(g['value'].count())
        # get global thresholds
        thr = thresholds.loc[(thresholds['cmdb_id']==cid)&(thresholds['trace_name']==tname)]
        global_p5 = float(thr['global_p5'].iloc[0]) if (not thr.empty and pd.notna(thr['global_p5'].iloc[0])) else pd.NA
        global_p95 = float(thr['global_p95'].iloc[0]) if (not thr.empty and pd.notna(thr['global_p95'].iloc[0])) else pd.NA

        if typ == 'traffic':
            min_val = float(g['value'].min())
            # earliest timestamp where value == min_val
            earliest_ts = g.loc[g['value'] == min_val, 'dt'].min()
            global_threshold_used = global_p5
            rows.append({
                'cmdb_id': cid,
                'trace_name': tname,
                'type': 'traffic',
                'window_count': window_count,
                'min_or_max_value': min_val,
                'earliest_breach_or_min_timestamp': earliest_ts,
                'global_threshold_used': global_threshold_used
            })
        elif typ == 'latency':
            max_val = float(g['value'].max())
            # earliest timestamp where value >= global_p95 (if global_p95 exists)
            if pd.isna(global_p95):
                earliest_breach = pd.NaT
            else:
                breaches = g.loc[g['value'] >= global_p95]
                earliest_breach = breaches['dt'].min() if not breaches.empty else pd.NaT
            global_threshold_used = global_p95
            rows.append({
                'cmdb_id': cid,
                'trace_name': tname,
                'type': 'latency',
                'window_count': window_count,
                'min_or_max_value': max_val,
                'earliest_breach_or_min_timestamp': earliest_breach,
                'global_threshold_used': global_threshold_used
            })
        else:
            # skip 'other' types per instruction (not requested), but include as fallback with NaNs
            rows.append({
                'cmdb_id': cid,
                'trace_name': tname,
                'type': 'other',
                'window_count': window_count,
                'min_or_max_value': float(g['value'].max()),
                'earliest_breach_or_min_timestamp': pd.NaT,
                'global_threshold_used': pd.NA
            })

    trace_window_summary = pd.DataFrame(rows)

    # sort by earliest_breach_timestamp ascending then by cmdb_id
    # ensure column exists and convert to datetime aware dtype if necessary
    if 'earliest_breach_or_min_timestamp' in trace_window_summary.columns:
        trace_window_summary['earliest_sort'] = pd.to_datetime(trace_window_summary['earliest_breach_or_min_timestamp'])
        trace_window_summary = trace_window_summary.sort_values(['earliest_sort','cmdb_id'], ascending=[True, True]).drop(columns=['earliest_sort'])
    else:
        trace_window_summary = trace_window_summary.sort_values(['cmdb_id'])
else:
    trace_window_summary = pd.DataFrame(columns=[
        'cmdb_id','trace_name','type','window_count','min_or_max_value',
        'earliest_breach_or_min_timestamp','global_threshold_used'
    ])

# limit to top 50 rows
trace_window_summary = trace_window_summary.head(50)

# 3) From filtered_error_logs, filter rows to same 09:52-09:54 window and cmdb_id in [apache01, apache02]
# Save as filtered_error_apache
if 'filtered_error_logs' in globals() and (not filtered_error_logs.empty):
    filtered_error_apache = filtered_error_logs.loc[
        (filtered_error_logs['dt'] >= start_window) &
        (filtered_error_logs['dt'] <= end_window) &
        (filtered_error_logs['cmdb_id'].isin(['apache01','apache02']))
    ].copy()
else:
    # If filtered_error_logs not present or empty, create empty DataFrame with expected columns
    filtered_error_apache = pd.DataFrame(columns=['timestamp','cmdb_id','message','dt'])

# Compute requested aggregates for filtered_error_apache
total_error_rows_apache = int(filtered_error_apache.shape[0])
if total_error_rows_apache > 0:
    earliest_error_apache = filtered_error_apache['dt'].min()
    latest_error_apache = filtered_error_apache['dt'].max()
    counts_per_cmdb_apache = filtered_error_apache.groupby('cmdb_id').size().sort_values(ascending=False).head(10).reset_index(name='count')
    # collect up to 50 message texts (full)
    messages_apache = filtered_error_apache['message'].astype(str).tolist()[:50]
    messages_df = pd.DataFrame({'message': messages_apache})
else:
    earliest_error_apache = pd.NaT
    latest_error_apache = pd.NaT
    counts_per_cmdb_apache = pd.DataFrame(columns=['cmdb_id','count'])
    messages_df = pd.DataFrame(columns=['message'])

# Keep trace_focus, trace_window_summary, filtered_error_apache in kernel variables for next steps (already set)

# Final compact outputs: trace_focus.shape (small), trace_window_summary (up to 50 rows), and filtered error summary aggregates
trace_focus_shape = trace_focus.shape if 'trace_focus' in globals() else (0,0)

trace_focus_shape, trace_window_summary, total_error_rows_apache, earliest_error_apache, latest_error_apache, counts_per_cmdb_apache, messages_df.head(50)

```
Out[5]:
```
```python
# Prepare a plain-English compact summary using existing kernel variables:
# trace_focus, trace_window_summary, filtered_error_apache

import pandas as pd

# Safely reference variables
tf = trace_focus if 'trace_focus' in globals() else pd.DataFrame()
tws = trace_window_summary if 'trace_window_summary' in globals() else pd.DataFrame()
fe_apache = filtered_error_apache if 'filtered_error_apache' in globals() else pd.DataFrame()

# Basic counts
trace_focus_rows = int(tf.shape[0])
trace_summary_rows = int(tws.shape[0])

# Earliest breach/min timestamp shown in the summary (if available)
if not tws.empty:
    earliest_breach = tws['earliest_breach_or_min_timestamp'].min()
else:
    earliest_breach = pd.NaT

# Quick diagnostic counts:
latency_with_breach = 0
traffic_below_threshold = 0
if not tws.empty:
    latency_with_breach = int(
        tws.loc[(tws['type']=='latency') & (pd.notna(tws['earliest_breach_or_min_timestamp'])), :].shape[0]
    )
    # traffic drop if min_or_max_value <= global_threshold_used (and threshold is not NA)
    mask_traffic = (tws['type']=='traffic') & pd.notna(tws['global_threshold_used'])
    traffic_below_threshold = int((tws.loc[mask_traffic, 'min_or_max_value'] <= tws.loc[mask_traffic, 'global_threshold_used']).sum())

# Error logs (apache) aggregates
error_count_apache = int(fe_apache.shape[0])
if error_count_apache > 0:
    err_earliest = fe_apache['dt'].min()
    err_latest = fe_apache['dt'].max()
    err_counts = fe_apache.groupby('cmdb_id').size().sort_values(ascending=False).reset_index(name='count').head(10)
    top_err_list = [(row['cmdb_id'], int(row['count'])) for _, row in err_counts.head(3).iterrows()]
    # collect up to 50 messages
    err_messages = fe_apache['message'].astype(str).tolist()[:50]
else:
    err_earliest = pd.NaT
    err_latest = pd.NaT
    err_counts = pd.DataFrame(columns=['cmdb_id','count'])
    top_err_list = []
    err_messages = []

# Build summary string
summary_lines = [
    "Focused trace window summary (2021-03-09 09:52:00+08 through 09:54:00+08):",
    f"- trace_focus contains {trace_focus_rows} rows from the selected components.",
    f"- trace_window_summary has {trace_summary_rows} series (top 50 rows returned).",
    f"- Earliest reported breach/min timestamp in the summary: {earliest_breach}",
    f"- Latency series with an actual breach in-window (earliest breach present): {latency_with_breach}",
    f"- Traffic series whose min in-window is <= the series' global p5 (indicative of traffic drop): {traffic_below_threshold}",
    "",
    "Notes on trace_window_summary content:",
    "- Each row shows cmdb_id, trace_name, type (traffic/latency), count in the 2-minute window, the min (traffic) or max (latency) value in-window,",
    "  the earliest timestamp of that min or of the first breach >= global p95, and the global threshold used.",
    "- The summary is sorted by earliest breach/min timestamp ascending then by cmdb_id.",
    "",
    "Error log (apache) findings in the same 2-minute window:",
    f"- Total error log rows for apache01/apache02 in-window: {error_count_apache}",
    f"- Earliest error timestamp (apache): {err_earliest}",
    f"- Latest error timestamp (apache): {err_latest}",
    f"- Top components by error-count (top 3): {top_err_list}",
    f"- Returned up to 50 error message texts (count returned: {len(err_messages)}).",
    "",
    "Stored kernel variables for follow-up:",
    "- trace_focus  (filtered trace rows for the 2-minute window and selected components)",
    "- trace_window_summary  (per-series compact summary for the window, up to 50 rows shown)",
    "- filtered_error_apache  (apache error log rows for the 2-minute window; up to 50 messages available)",
    "",
    "If you want, I can (a) show the top 20 rows of trace_window_summary, (b) list the top error messages, or (c) filter to a specific component/trace."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((510, 5),       cmdb_id                        trace_name     type  window_count  min_or_max_value earliest_breach_or_min_timestamp  global_threshold_used
22       MG01  trace.from_Tomcat04.duration_p95  latency             3          2.339000        2021-03-09 09:52:00+08:00               2.223172
26       MG01     trace.from_dockerA1.row_count  traffic             3          3.000000        2021-03-09 09:52:00+08:00               7.000000
29       MG01     trace.from_dockerA2.row_count  traffic             3          5.000000        2021-03-09 09:52:00+08:00               7.450000
35       MG01     trace.from_dockerB2.row_count  traffic             3        157.000000        2021-03-09 09:52:00+08:00              57.800000
40       MG01           trace.self.duration_p95  latency             3          2.088000        2021-03-09 09:52:00+08:00               2.054710
..        ...                               ...      ...           ...               ...                              ...                    ...
102  Tomcat01          trace.self.duration_mean  latency             3          0.083292        2021-03-09 09:53:00+08:00               0.083292
103  Tomcat01           trace.self.duration_p95  latency             3          1.004000        2021-03-09 09:53:00+08:00               1.004000
104  Tomcat01              trace.self.row_count  traffic             3       2333.000000        2021-03-09 09:53:00+08:00            2577.500000
107  Tomcat01           trace.to_MG01.row_count  traffic             3        171.000000        2021-03-09 09:53:00+08:00             191.800000
110  Tomcat01           trace.to_MG02.row_count  traffic             3        175.000000        2021-03-09 09:53:00+08:00             177.250000

[50 rows x 7 columns], 45, Timestamp('2021-03-09 09:53:26+0800', tz='Asia/Shanghai'), Timestamp('2021-03-09 09:53:42+0800', tz='Asia/Shanghai'),     cmdb_id  count
0  apache02     26
1  apache01     19,                                               message
0   IPAddress "POST /UOCP/base/ServiceTest7.json H...
1   IPAddress "POST /UOCP/person/ServiceTest3.json...
2   IPAddress "POST /UOCP/person/ServiceTest10.jso...
3   IPAddress "POST /UOCP/person/ServiceTest3.json...
4   IPAddress "POST /UOCP/base/ServiceTest2.json H...
..                                                ...
40  IPAddress "POST /UOCP/base/ServiceTest6.json H...
41  IPAddress "POST /UOCP/base/ServiceTest4.json H...
42  IPAddress "POST /UOCP/base/ServiceTest6.json H...
43  IPAddress "POST /UOCP/base/ServiceTest4.json H...
44  IPAddress "POST /UOCP/person/ServiceTest10.jso...

[45 rows x 1 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```